In [1]:
!pip -q install catboost scikit-learn pandas numpy matplotlib joblib

In [3]:
import pandas as pd, numpy as np
url = "https://raw.githubusercontent.com/selva86/datasets/master/GermanCredit.csv"
df = pd.read_csv(url)
print(df.shape)
print(df.columns.tolist())

(1000, 21)
['status', 'duration', 'credit_history', 'purpose', 'amount', 'savings', 'employment_duration', 'installment_rate', 'personal_status_sex', 'other_debtors', 'present_residence', 'property', 'age', 'other_installment_plans', 'housing', 'number_credits', 'job', 'people_liable', 'telephone', 'foreign_worker', 'credit_risk']


In [5]:
from sklearn.preprocessing import LabelEncoder
cols = df.columns.tolist()
candidates = [c for c in cols if any(k in c.lower() for k in ['credit','class','target','default','creditability','good','bad'])]
target = candidates[0] if candidates else None
if target is None:
    bin_cols = [c for c in cols if pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique()<=3]
    target = bin_cols[0] if bin_cols else cols[-1]
y_raw = df[target]
uniq = sorted(pd.unique(y_raw))
if set(uniq)==set([1,2]):
    y = (y_raw==2).astype(int)
elif set(uniq)==set([0,1]):
    y = y_raw.astype(int)
else:
    low = y_raw.astype(str).str.lower()
    if any('bad' in s for s in low):
        y = low.str.contains('bad').astype(int)
    elif any('good' in s for s in low):
        y = (~low.str.contains('good')).astype(int)
    else:
        y = LabelEncoder().fit_transform(y_raw)
id_cols = [c for c in df.columns if 'id' in c.lower()]
X = df.drop(columns=[target] + id_cols)
print('target:', target, '-> mapped to 0/1')

target: credit_history -> mapped to 0/1


In [7]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [9]:
num_cols = X_train.select_dtypes(include=['number']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object','category']).columns.tolist()
print(len(num_cols), 'numeric,', len(cat_cols), 'categorical')

7 numeric, 12 categorical


In [11]:
print("Unique target values:", np.unique(y))
print("Counts:", np.bincount(y))


Unique target values: [0 1 2 3 4]
Counts: [ 49 293  88 530  40]


In [13]:
y = df[target].apply(lambda v: 0 if v in [0,1] else 1)

print("Fixed target values:", np.unique(y, return_counts=True))

    

Fixed target values: (array([1], dtype=int64), array([1000], dtype=int64))


In [15]:
y = df[target].apply(lambda v: 1 if v in [0,1] else 0)

print("Fixed target values:", np.unique(y, return_counts=True))

Fixed target values: (array([0], dtype=int64), array([1000], dtype=int64))


In [17]:

import pandas as pd

url = "https://raw.githubusercontent.com/selva86/datasets/master/GermanCredit.csv"
df = pd.read_csv(url)

print(df.columns)  # check all columns
print(df['credit_risk'].value_counts())  # target distribution

# Map to binary: Good = 0, Bad = 1
y = df['credit_risk'].map({'good':0, 'bad':1})
X = df.drop(columns=['credit_risk'])

print("Target values:", y.unique())
print("Good/Bad counts:", y.value_counts())

Index(['status', 'duration', 'credit_history', 'purpose', 'amount', 'savings',
       'employment_duration', 'installment_rate', 'personal_status_sex',
       'other_debtors', 'present_residence', 'property', 'age',
       'other_installment_plans', 'housing', 'number_credits', 'job',
       'people_liable', 'telephone', 'foreign_worker', 'credit_risk'],
      dtype='object')
credit_risk
1    700
0    300
Name: count, dtype: int64
Target values: [nan]
Good/Bad counts: Series([], Name: count, dtype: int64)


In [19]:
y = df['credit_risk']
X = df.drop(columns=['credit_risk'])

print("Unique target values:", y.unique())
print("Counts:\n", y.value_counts())


Unique target values: [1 0]
Counts:
 credit_risk
1    700
0    300
Name: count, dtype: int64


In [29]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import numpy as np

# Build pipeline
pipe_lr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500))
])

# Fit the pipeline
pipe_lr.fit(X_train.select_dtypes("number"), y_train)

# Predict probabilities
proba_lr = pipe_lr.predict_proba(X_val.select_dtypes("number"))

# Compute AUC
if len(np.unique(y_val)) == 2:
    # Binary classification
    auc = roc_auc_score(y_val, proba_lr[:,1])
else:
    # Multiclass classification
    auc = roc_auc_score(y_val, proba_lr, multi_class='ovr')

print("LR AUC:", round(auc, 4))

LR AUC: 0.7808


In [32]:
cat_cols = X_train.select_dtypes(include="object").columns.tolist()

train_pool = Pool(X_train, y_train, cat_features=cat_cols)
val_pool   = Pool(X_val, y_val, cat_features=[c for c in cat_cols if c in X_val.columns])

cbc = CatBoostClassifier(iterations=200, learning_rate=0.1, depth=6, verbose=0)
cbc.fit(train_pool, eval_set=val_pool, use_best_model=True)

proba_cb = cbc.predict_proba(val_pool)[:,1]

import numpy as np
from sklearn.metrics import roc_auc_score

if len(np.unique(y_val)) == 2:
    auc = roc_auc_score(y_val, proba_cb)
else:
    proba_cb_all = cbc.predict_proba(val_pool)
    auc = roc_auc_score(y_val, proba_cb_all, multi_class='ovr')

print("CatBoost AUC:", round(auc, 4))

CatBoost AUC: 0.807


In [25]:
import numpy as np

COST_FP, COST_FN = 1, 5

def best_threshold(y_true, y_proba):
    best = (None, 1e18)
    for t in np.linspace(0.01,0.99,99):
        y_pred = (y_proba >= t).astype(int)
        fp = np.sum((y_true==0) & (y_pred==1))
        fn = np.sum((y_true==1) & (y_pred==0))
        cost = fp*COST_FP + fn*COST_FN
        if cost < best[1]: best = (t, cost)
    return best

print("LR best threshold & cost:", best_threshold(y_val, proba_lr))

LR best threshold & cost: (0.01, 10)


In [33]:
proba_cb = cbc.predict_proba(val_pool)[:,1]

In [36]:
from sklearn.metrics import roc_auc_score

def best_threshold(y_true, y_proba):
    """Find threshold that maximizes Youden's J statistic (TPR - FPR)."""
    from sklearn.metrics import roc_curve
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    J = tpr - fpr
    idx = J.argmax()
    return thresholds[idx], J[idx]

In [40]:
t_lr, _ = best_threshold(y_val, proba_lr)
t_cb, _ = best_threshold(y_val, proba_cb)

print("\n--- Logistic Regression ---")
print(confusion_matrix(y_val, (proba_lr>=t_lr).astype(int)))
print(classification_report(y_val, (proba_lr>=t_lr).astype(int)))

print("\n--- CatBoost ---")
print(confusion_matrix(y_val, (proba_cb>=t_cb).astype(int)))
print(classification_report(y_val, (proba_cb>=t_cb).astype(int)))

ValueError: multiclass format is not supported